# Liquidity Absorption 09 — Observable lifecycle of the original two-touch signal

**Study question:** What happens from **level becomes available → first completed touch → second completed touch/signal → market reaction**? Instead of projecting an eventual pivot back into the past, we use the original A02 five-minute pivot confirmation delay (two right-side 5m candles, **10 minutes after the pivot candle close**). Prior-day levels become available at **next UTC midnight**. All stage boundaries are completed **1-minute Binance USDT-M candles**: the intraminute touch instant is unknown.

**Primary outcome:** frozen **A03b fixed gross 1:1** TARGET / STOP / TIMEOUT, one row per 452 original candidate IDs. A03c maker third-touch outcomes are **eligibility annotations only**, never live fills. These historical July–September 2026 outcomes have been inspected repeatedly: every TARGET-vs-STOP contrast in this notebook is descriptive and *not* out-of-sample evidence.

The 60 minutes before the retrospective level source show only general market context; the source→availability phase captures unconfirmed pivot bars or the forming previous UTC day. **Distance to the eventually selected level is never exposed as a contemporaneously available feature before availability.** After exit the market can continue moving; this is **not trade P&L**. No signals, stops, targets, collectors, or source results are modified.


In [ ]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display,Image
REPO=Path.cwd().resolve()
while REPO!=REPO.parent and not (REPO/'src'/'data_engineering').is_dir(): REPO=REPO.parent
assert (REPO/'src'/'data_engineering').is_dir(), 'Open Jupyter from the repository'
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a09 import VERSION,Config,load_cohort,run
assert VERSION=='LA-LIFECYCLE-09-0.1.0', 'Pull main and restart kernel'
t=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_liquidity_absorption_stage_a09.py','-q'],capture_output=True,text=True,cwd=REPO)
print(t.stdout[-1000:],t.stderr[-2000:]);assert t.returncode==0,'LA09 regression suite failed'
BINANCE=binance_root(REPO)
OUT=REPO/'notebooks'/'Liquidity_Absorption'/'09'
print('Version:',VERSION,'Market archive:',BINANCE,'Output:',OUT)


## 1. Audit when the selected level actually becomes observable

The original A02 detector uses a 5m pivot with **two subsequent completed 5m candles** (10-minute visibility lag), or a previous fully completed UTC-day high/low (visible at following midnight). Level-source time is **not** necessarily the same as availability time. First and second touch timestamps are **completed 1m candle closes**, so the exact order inside each candle is not inferred. Confirm that all first touches follow level availability.


In [ ]:
cohort=load_cohort(REPO)
print('Original signal population',len(cohort))
display(cohort.groupby(['level_type','reason']).size().rename('n').unstack(fill_value=0))
display(cohort.groupby('level_type')[['minutes_source_to_available','minutes_available_to_first','minutes_first_to_second']].agg(['min','median','max']).round(2))
assert (cohort.available_ms<=cohort.first_touch_close_ms).all()
print('First-touch closes exactly at readiness (intraminute order unknown):',int(cohort.first_touch_confirms_at_availability.sum()))


## 2. Extract the complete, stage-aligned market anatomy

**Eight predeclared stages:** 60m before eventual source (generic context only); source→availability (unconfirmed); availability→first touch; first→second touch; post-signal +1..+5, +6..+15, +16..+60, +61..+240. Stage observations use completed minute bars in disjoint half-open time intervals. The first-touch and signal one-minute candles are included in their correct formation stage. Missing 1m bars invalidate their corresponding stage—never forward-filled. Also save 4 × 46-minute landmark-aligned micro-panels around **source**, **availability**, **first touch**, and **second touch**. All groups are retrospective outcome descriptions; not a new signal filter.


In [ ]:
decision=run(REPO,BINANCE,OUT,Config())
display(decision)


## 3. Formation-phase comparisons by underlying, level family and eventual outcome

Check duration, directional price change, executed aggressive-flow proxy and zone-intersection rate. **The stage-lengths are variable** in the level-available→first and first→signal intervals; report each stage's duration rather than interpreting aggregate quote amount alone. A03b gross 1:1 target/stop was not re-optimized. The A03c maker outcome column is not evidence of a filled passive order.


In [ ]:
stages=pd.read_csv(OUT/'09_lifecycle_stages.csv')
summary=pd.read_csv(OUT/'09_lifecycle_summary.csv')
print('Incomplete or empty stages (must not enter medians):')
display(stages.loc[~stages.complete].groupby(['symbol','stage']).size().rename('n'))
show=['symbol','level_type','eventual_a03b_outcome','stage','n_events','median_duration_min',
      'median_signed_price_change_bp','median_aligned_aggressive_quote_fraction',
      'median_zone_intersect_fraction']
display(summary.loc[summary.level_type.eq('ALL_LEVELS'),show].round(3))


## 4. Compare three observable clocks, not one eventual signal timestamp

The four alignment panels are **−15..+30 complete minutes** around level source, level availability, first touch and second-touch confirmation. For pivot levels, all relative minutes before availability have **undefined level-distance/zone variables** by construction. Post-exit price is market context only and must not be interpreted as continuing strategy returns.


In [ ]:
aligned=pd.read_csv(OUT/'09_landmark_summary.csv')
display(aligned.query("level_type=='ALL_LEVELS' and relative_min in [-15, -5, -1, 0, 1, 5, 15, 30]")[['symbol','milestone','outcome','relative_min','n_events','n_post_exit','median_signed_price_bp','median_aligned_flow']].round(3))
for name in ('09_milestone_price_paths.png','09_lifecycle_phase_changes.png'):
    display(Image(filename=str(OUT/name)))


## 5. Interpretation and next research gate

A completed two-touch rejection is constructed to involve an approach and rejection. **Differences between the historical winner/loser groups are conditional on already having a signal**; an absent no-signal control prevents us from claiming what *causes* a signal to appear. To study signal formation probability we will need an eligible-but-not-second-touched risk-set comparison generated at level availability—not select controls based on future confirmation. Any candidate pattern here needs fresh unexamined dates and matched geometry, side, symbol and level type before prospective testing. The 2024–2026 Claude EXT/W01 results are already inspected as well and cannot honestly be called pristine confirmation.
